In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
from pathlib import Path

from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
import os
os.getcwd()

'/Users/azizulshaikh/Projects/Simple-RAG-Project'

In [7]:
os.chdir("/Users/azizulshaikh/Projects/Simple-RAG-Project/")

In [12]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx"}
folder_path = "data/documents"

In [13]:
folder = Path(folder_path)
if not folder.exists():
    raise FileNotFoundError(f"Document folder not found: {folder_path}")
else:
    print(f"Document folder found: {folder_path}")

Document folder found: data/documents


In [18]:
sorted(folder.iterdir())

[PosixPath('data/documents/company_handbook.pdf'),
 PosixPath('data/documents/refund_policy.pdf'),
 PosixPath('data/documents/remote_work_guidelines.docx')]

In [30]:
# Count Source Files in data/documents folder
sum(
        1
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

3

In [17]:
sorted(folder.iterdir())[0].is_dir()

False

In [16]:
sorted(folder.iterdir())[0].is_file()

True

In [19]:
files = [
    path
    for path in sorted(folder.iterdir())
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
]

In [20]:
files

[PosixPath('data/documents/company_handbook.pdf'),
 PosixPath('data/documents/refund_policy.pdf'),
 PosixPath('data/documents/remote_work_guidelines.docx')]

In [21]:
str(files[0])

'data/documents/company_handbook.pdf'

In [24]:
files[0].name

'company_handbook.pdf'

In [22]:
# loading each document from filepaths in files variable
file_path = files[0]
suffix = file_path.suffix.lower()

try:
    if suffix == ".pdf":
        # PyPDFLoader creates one Document per page and includes page numbers.
        loader = PyPDFLoader(str(file_path))
        docs = loader.load()
    elif suffix == ".docx":
        # DOCX loaders usually do not provide page numbers.
        loader = Docx2txtLoader(str(file_path))
        docs = loader.load()
    else:
        raise ValueError(
            f"Unsupported file type: {file_path.name}. "
            "Only PDF and DOCX files are supported."
        )
except ValueError:
    raise
except Exception as exc:
    raise ValueError(
        f"Could not read '{file_path.name}'. "
        "The file may be invalid or corrupted."
    ) from exc

In [23]:
docs

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'data/documents/company_handbook.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 days into\nthe next yea

In [25]:
file_path.name

'company_handbook.pdf'

In [26]:
for doc in docs:
    doc.metadata["source"] = file_path.name
    doc.metadata["filename"] = file_path.name
    # PDF loaders usually set "page" (0-based). Convert to 1-based for display.
    if "page" in doc.metadata and doc.metadata["page"] is not None:
        try:
            doc.metadata["page"] = int(doc.metadata["page"]) + 1
        except (TypeError, ValueError):
            doc.metadata["page"] = None
    else:
        doc.metadata["page"] = None

In [27]:
docs

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'company_handbook.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'filename': 'company_handbook.pdf'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 da

In [28]:
documents: list[Document]=[]

documents.extend(docs)
documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'company_handbook.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'filename': 'company_handbook.pdf'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 da

In [29]:
documents = [doc for doc in documents if doc.page_content and doc.page_content.strip()]
documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'company_handbook.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'filename': 'company_handbook.pdf'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 da

In [32]:
# split documents

if not documents:
    raise ValueError("No documents to split.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", 1000)),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", 200)),
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(documents)

chunks

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'company_handbook.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'filename': 'company_handbook.pdf'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 da